In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.utils import resample
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_B.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_B" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_B")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:
        return 3  # 広告経由の購入
    elif row["event_type"] == 2:
        return 2  # 広告クリック
    elif row["event_type"] == 1:
        return 1  # 詳細ページ閲覧
    else:
        return 0  # カート追加

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [5]:
# タイムスタンプ処理
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [6]:
# ユーザーと商品の特徴量
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [7]:
# **新しい特徴量（ユーザー・商品間の関係性）**
interaction_features = train_data.groupby(["user_id", "product_id"]).agg(
    user_product_interaction_count=("event_type", "count"),
    user_product_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")
train_data = train_data.merge(interaction_features, on=["user_id", "product_id"], how="left")

In [9]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users",
    "user_product_interaction_count", "user_product_avg_time_since"
]

In [10]:
# データバランス調整（リサンプリング最適化）
train_data_pos = train_data[train_data["relevance"] >= 2]  
train_data_neg = train_data[train_data["relevance"] < 2]   

train_data_neg_undersampled = resample(train_data_neg, replace=True, n_samples=700000, random_state=42)
train_data_pos_oversampled = resample(train_data_pos, replace=True, n_samples=800000, random_state=42)

train_data_balanced = pd.concat([train_data_neg_undersampled, train_data_pos_oversampled])

In [11]:
# クロスバリデーション
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ndcg_scores = []

In [12]:
for train_index, val_index in kf.split(train_data_balanced):
    train_set = train_data_balanced.iloc[train_index]
    val_set = train_data_balanced.iloc[val_index]

    X_train, y_train = train_set[features], train_set["relevance"]
    X_val, y_val = val_set[features], val_set["relevance"]

    group_train = train_set.groupby("user_id").size().to_numpy()
    group_val = val_set.groupby("user_id").size().to_numpy()

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(group_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(group_val)

    params = {
        "objective": "rank:ndcg",
        "eval_metric": "ndcg",
        "booster": "gbtree",
        "eta": 0.05,  # 学習率を下げて汎化性能向上
        "max_depth": 7,
        "min_child_weight": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "lambda": 1.5,
        "gamma": 0.2
    }

    evals_result = {}
    model = xgb.train(params, dtrain, num_boost_round=500,
                      evals=[(dtrain, "train"), (dval, "val")],
                      evals_result=evals_result,
                      early_stopping_rounds=30,
                      verbose_eval=10)

    val_set_copy = val_set.copy()
    val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

    def ndcg_at_k(relevance_scores, k):
        relevance_scores = np.array(relevance_scores)
        ideal_relevance = np.sort(relevance_scores)[::-1]  # 降順ソート

        # kに満たない場合、ゼロ埋め
        if len(relevance_scores) < k:
            relevance_scores = np.pad(relevance_scores, (0, k - len(relevance_scores)), constant_values=0)
            ideal_relevance = np.pad(ideal_relevance, (0, k - len(ideal_relevance)), constant_values=0)

        dcg = np.sum(relevance_scores[:k] / np.log2(np.arange(2, k + 2)))
        ideal_dcg = np.sum(ideal_relevance[:k] / np.log2(np.arange(2, k + 2)))

        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()
    ndcg_scores.append(ndcg_val)

[0]	train-ndcg:0.97146	val-ndcg:0.97875
[10]	train-ndcg:0.97876	val-ndcg:0.98225
[20]	train-ndcg:0.98011	val-ndcg:0.98290
[30]	train-ndcg:0.98061	val-ndcg:0.98345
[40]	train-ndcg:0.98116	val-ndcg:0.98378
[50]	train-ndcg:0.98192	val-ndcg:0.98427
[60]	train-ndcg:0.98257	val-ndcg:0.98461
[70]	train-ndcg:0.98296	val-ndcg:0.98494
[80]	train-ndcg:0.98327	val-ndcg:0.98508
[90]	train-ndcg:0.98362	val-ndcg:0.98539
[100]	train-ndcg:0.98389	val-ndcg:0.98550
[110]	train-ndcg:0.98431	val-ndcg:0.98578
[120]	train-ndcg:0.98464	val-ndcg:0.98601
[130]	train-ndcg:0.98495	val-ndcg:0.98619
[140]	train-ndcg:0.98530	val-ndcg:0.98641
[150]	train-ndcg:0.98552	val-ndcg:0.98667
[160]	train-ndcg:0.98577	val-ndcg:0.98685
[170]	train-ndcg:0.98595	val-ndcg:0.98696
[180]	train-ndcg:0.98634	val-ndcg:0.98724
[190]	train-ndcg:0.98663	val-ndcg:0.98753
[200]	train-ndcg:0.98699	val-ndcg:0.98777
[210]	train-ndcg:0.98719	val-ndcg:0.98788
[220]	train-ndcg:0.98739	val-ndcg:0.98799
[230]	train-ndcg:0.98762	val-ndcg:0.98814
[24

/tmp/ipykernel_24208/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.97168	val-ndcg:0.97926
[10]	train-ndcg:0.97836	val-ndcg:0.98264
[20]	train-ndcg:0.97949	val-ndcg:0.98347
[30]	train-ndcg:0.98000	val-ndcg:0.98365
[40]	train-ndcg:0.98058	val-ndcg:0.98401
[50]	train-ndcg:0.98136	val-ndcg:0.98455
[60]	train-ndcg:0.98217	val-ndcg:0.98507
[70]	train-ndcg:0.98270	val-ndcg:0.98524
[80]	train-ndcg:0.98311	val-ndcg:0.98551
[90]	train-ndcg:0.98343	val-ndcg:0.98577
[100]	train-ndcg:0.98369	val-ndcg:0.98590
[110]	train-ndcg:0.98397	val-ndcg:0.98611
[120]	train-ndcg:0.98436	val-ndcg:0.98644
[130]	train-ndcg:0.98474	val-ndcg:0.98669
[140]	train-ndcg:0.98516	val-ndcg:0.98702
[150]	train-ndcg:0.98549	val-ndcg:0.98725
[160]	train-ndcg:0.98590	val-ndcg:0.98751
[170]	train-ndcg:0.98621	val-ndcg:0.98770
[180]	train-ndcg:0.98662	val-ndcg:0.98801
[190]	train-ndcg:0.98698	val-ndcg:0.98826
[200]	train-ndcg:0.98735	val-ndcg:0.98856
[210]	train-ndcg:0.98765	val-ndcg:0.98875
[220]	train-ndcg:0.98789	val-ndcg:0.98894
[230]	train-ndcg:0.98808	val-ndcg:0.98909
[24

/tmp/ipykernel_24208/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.97185	val-ndcg:0.97901
[10]	train-ndcg:0.97806	val-ndcg:0.98195
[20]	train-ndcg:0.97938	val-ndcg:0.98249
[30]	train-ndcg:0.97977	val-ndcg:0.98262
[40]	train-ndcg:0.98049	val-ndcg:0.98304
[50]	train-ndcg:0.98129	val-ndcg:0.98363
[60]	train-ndcg:0.98183	val-ndcg:0.98400
[70]	train-ndcg:0.98248	val-ndcg:0.98442
[80]	train-ndcg:0.98311	val-ndcg:0.98480
[90]	train-ndcg:0.98361	val-ndcg:0.98508
[100]	train-ndcg:0.98394	val-ndcg:0.98526
[110]	train-ndcg:0.98426	val-ndcg:0.98554
[120]	train-ndcg:0.98456	val-ndcg:0.98569
[130]	train-ndcg:0.98499	val-ndcg:0.98603
[140]	train-ndcg:0.98534	val-ndcg:0.98628
[150]	train-ndcg:0.98566	val-ndcg:0.98655
[160]	train-ndcg:0.98596	val-ndcg:0.98671
[170]	train-ndcg:0.98627	val-ndcg:0.98693
[180]	train-ndcg:0.98665	val-ndcg:0.98722
[190]	train-ndcg:0.98700	val-ndcg:0.98756
[200]	train-ndcg:0.98733	val-ndcg:0.98779
[210]	train-ndcg:0.98756	val-ndcg:0.98800
[220]	train-ndcg:0.98784	val-ndcg:0.98816
[230]	train-ndcg:0.98802	val-ndcg:0.98835
[24

/tmp/ipykernel_24208/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.97254	val-ndcg:0.97890
[10]	train-ndcg:0.97834	val-ndcg:0.98204
[20]	train-ndcg:0.97865	val-ndcg:0.98189
[30]	train-ndcg:0.97971	val-ndcg:0.98244
[40]	train-ndcg:0.98045	val-ndcg:0.98298
[50]	train-ndcg:0.98122	val-ndcg:0.98347
[60]	train-ndcg:0.98198	val-ndcg:0.98405
[70]	train-ndcg:0.98254	val-ndcg:0.98441
[80]	train-ndcg:0.98315	val-ndcg:0.98477
[90]	train-ndcg:0.98360	val-ndcg:0.98503
[100]	train-ndcg:0.98407	val-ndcg:0.98532
[110]	train-ndcg:0.98444	val-ndcg:0.98557
[120]	train-ndcg:0.98484	val-ndcg:0.98576
[130]	train-ndcg:0.98513	val-ndcg:0.98601
[140]	train-ndcg:0.98551	val-ndcg:0.98629
[150]	train-ndcg:0.98580	val-ndcg:0.98651
[160]	train-ndcg:0.98623	val-ndcg:0.98683
[170]	train-ndcg:0.98656	val-ndcg:0.98711
[180]	train-ndcg:0.98693	val-ndcg:0.98738
[190]	train-ndcg:0.98720	val-ndcg:0.98757
[200]	train-ndcg:0.98739	val-ndcg:0.98770
[210]	train-ndcg:0.98766	val-ndcg:0.98790
[220]	train-ndcg:0.98794	val-ndcg:0.98814
[230]	train-ndcg:0.98814	val-ndcg:0.98827
[24

/tmp/ipykernel_24208/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.97210	val-ndcg:0.97899
[10]	train-ndcg:0.97827	val-ndcg:0.98169
[20]	train-ndcg:0.97908	val-ndcg:0.98208
[30]	train-ndcg:0.97933	val-ndcg:0.98230
[40]	train-ndcg:0.97990	val-ndcg:0.98264
[50]	train-ndcg:0.98079	val-ndcg:0.98323
[60]	train-ndcg:0.98155	val-ndcg:0.98372
[70]	train-ndcg:0.98234	val-ndcg:0.98428
[80]	train-ndcg:0.98297	val-ndcg:0.98464
[90]	train-ndcg:0.98343	val-ndcg:0.98490
[100]	train-ndcg:0.98394	val-ndcg:0.98532
[110]	train-ndcg:0.98440	val-ndcg:0.98563
[120]	train-ndcg:0.98473	val-ndcg:0.98584
[130]	train-ndcg:0.98505	val-ndcg:0.98611
[140]	train-ndcg:0.98535	val-ndcg:0.98631
[150]	train-ndcg:0.98566	val-ndcg:0.98653
[160]	train-ndcg:0.98596	val-ndcg:0.98667
[170]	train-ndcg:0.98628	val-ndcg:0.98686
[180]	train-ndcg:0.98663	val-ndcg:0.98712
[190]	train-ndcg:0.98701	val-ndcg:0.98741
[200]	train-ndcg:0.98730	val-ndcg:0.98762
[210]	train-ndcg:0.98754	val-ndcg:0.98779
[220]	train-ndcg:0.98775	val-ndcg:0.98798
[230]	train-ndcg:0.98793	val-ndcg:0.98811
[24

/tmp/ipykernel_24208/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


In [13]:
print(f"Average nDCG@22 from CV: {np.mean(ndcg_scores):.4f}")

Average nDCG@22 from CV: 0.9170


In [14]:
# テストデータの処理
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [15]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [16]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")
test_data = test_data.merge(interaction_features, on=["user_id", "product_id"], how="left")
test_data.fillna(0, inplace=True)

In [17]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [18]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [19]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [20]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

def evaluate_ndcg(data, ground_truth, k=22):
    ndcg_scores = []
    for user_id, group in data.groupby("user_id"):
        predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
        relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
        ndcg_scores.append(ndcg_at_k(relevance_scores, k))
    return np.mean(ndcg_scores) if ndcg_scores else 0

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.0942


In [21]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]
submission['rank'] = submission['rank'].astype(int)

In [22]:
# 保存
submission.to_csv('./predict_file/predictions_B.tsv', sep="\t", index=False)